# Genixcure · PCR 11개 Existing Unique CSV Primer-aware Trimming & Recount

**입력:** 기존 11개 `*_merged_unique_sequences.csv.gz` (`sequence,read_count,length_bp`); FASTQ 재다운로드 또는 fastp 재병합 **불필요**.

**주의:** 실제 CSV를 사전 점검해 보면 AAV Forward primer의 역상보 서열(`GCCACTCCCACTGTCCTTTC`)이 3′ 말단에 자주 있으며, 다른 primer는 상대적으로 적습니다. 따라서 양쪽 primer를 확인한 결과와 한쪽만 확인한 결과를 **반드시 분리**합니다. Primer 존재가 확인되지 않은 6-base prefix를 무조건 Index로 간주해 제거하지 않습니다. 중간에 있는 motif는 제거하지 않습니다.

- `both_end`: 양끝에 검증된 Primer가 있고 두 Primer를 제거해 남은 서열. **가장 보수적인 trimmed 결과**.
- `single_end`: 한쪽 Primer만 검출되어 해당 쪽만 절단한 서열. 반대쪽은 미확인.
- `unmatched`: 양쪽 Primer 미검출, 원본 서열 유지.
- `ambiguous`: 방향/Primer 절단 위치가 충돌하거나 절단 후 서열이 없는 경우; 원본 유지.

각 분류는 **별도 CSV.gz 파일**, 같은 분류 내 정확히 동일한 서열의 read_count를 재합산합니다. 분류 간 임의 병합은 하지 않습니다. 원본 `All_10_Samples` 폴더는 읽기만 하며, 결과는 새 `Primer_Trimming_v1` 폴더에 저장됩니다. 출력의 `read_count` 총합(4개 분류 합계)은 입력 merged_reads와 같아야 합니다.

이 분석은 알려진 Primer의 **완전 일치 서열**만 대상으로 하며, truncated/mismatch primer는 남을 수 있습니다. 실제 라이브러리 구조를 모르는 상태에서 P5/P7 adapter / 6N barcode의 경계를 추측해 제거하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import csv, gzip, os, subprocess, shutil, time, collections, json
import pandas as pd

BASE=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/All_10_Samples')
OUT=Path('/content/drive/MyDrive/Genixcure_PCR_Merge_Analysis/Primer_Trimming_v1')
WORK=Path('/content/genixcure_primer_trim_work')
OUT.mkdir(parents=True,exist_ok=True); WORK.mkdir(parents=True,exist_ok=True)
SAMPLES=['Brain_1','Brain_2','Brain_3','Motor_1','Motor_2','Motor_3','SH_SY5Y_1','SH_SY5Y_2','SH_SY5Y_3','Input','PCR_6_13']
MAX_END_OVERHANG=45  # Read 종단에서 45 nt 이내에 완전 일치하는 primer만 trimming
SORT_RAM='2G'
OVERWRITE=False
print('Available:',sum((BASE/f'{s}_merged_unique_sequences.csv.gz').exists() for s in SAMPLES),'/',len(SAMPLES))


## 1. Primer 및 방향 정의

고객이 제공한 AAV primer를 기반으로 정방향/역상보를 탐색합니다. 5′ 말단에서는 F 또는 R, 3′ 말단에서는 그 역상보를 검사합니다. 5′의 F/R는 5′ 방향에서, 3′의 RC(F)/RC(R)는 3′ 방향에서 제거합니다. 방향을 역상보로 통일하는 것은 **이번 분석에서 수행하지 않습니다**. 양끝이 동일한 primer를 보이는 경우도 분류에는 남기지만, 올바른 library architecture는 추가 확인이 필요합니다.


In [ ]:
PRIMERS={'AAV_F':'GAAAGGACAGTGGGAGTGGC','AAV_R':'GCATGGACGAGCTGTACAAG'}
TRANSLATION=str.maketrans('ACGT','TGCA')
def rc(seq): return seq.translate(TRANSLATION)[::-1]

def detect_terminal(seq):
    # Primer motif를 내부에서 찾았다는 이유만으로 자르지 않음.
    # 앞/뒤에 완전 일치하는 motif가 여러 개면 높은 confidence로 취급하지 않음.
    left=[]; right=[]
    for name,motif in PRIMERS.items():
        pos=seq.find(motif)
        while pos>=0 and pos<=MAX_END_OVERHANG:
            left.append((name,pos,pos+len(motif)))
            pos=seq.find(motif,pos+1)
        rev=rc(motif)
        pos=seq.find(rev)
        while pos>=0:
            if len(seq)-(pos+len(rev))<=MAX_END_OVERHANG:
                right.append((name+'_RC',pos,pos+len(rev)))
            pos=seq.find(rev,pos+1)
    # Terminal에 가장 가까운 motif만 허용. 서로 다른 motif가 동률로 걸리면 ambiguous.
    if left:
        nearest=min(x[1] for x in left)
        left=[x for x in left if x[1]==nearest]
    if right:
        nearest=min(len(seq)-x[2] for x in right)
        right=[x for x in right if len(seq)-x[2]==nearest]
    if len(left)>1 or len(right)>1:
        return 'ambiguous',seq,'multiple_primer_candidates'
    a=left[0] if left else None; b=right[0] if right else None
    if a and b and a[2]>b[1]:
        return 'ambiguous',seq,'overlapping_primer_spans'
    if not a and not b:
        return 'unmatched',seq,'none'
    start=a[2] if a else 0
    end=b[1] if b else len(seq)
    if start>=end:
        return 'ambiguous',seq,'empty_after_trim'
    status='both_end' if a and b else 'single_end'
    pattern=(a[0] if a else 'none')+'__'+(b[0] if b else 'none')
    return status,seq[start:end],pattern

# Self-tests: annotated F + insert + reverse-complement R, single right, no match
assert detect_terminal(PRIMERS['AAV_F']+'AACCGGTT'+rc(PRIMERS['AAV_R']))[:2]==('both_end','AACCGGTT')
assert detect_terminal('AACCGGTT'+rc(PRIMERS['AAV_F']))[:2]==('single_end','AACCGGTT')
assert detect_terminal('ACGTACGT')[:2]==('unmatched','ACGTACGT')
print('Primer orientation self-tests PASS')


## 2. 스트리밍 변환 + 외부 sort 기반 재집계

6,335,288 Unique 행을 포함한 PCR_6_13도 전체 CSV를 DataFrame으로 RAM에 읽지 않습니다. 원본 각 행의 `read_count`를 중간 TSV로 내보내고, GNU `sort`와 streaming group-by로 Count를 합칩니다. 샘플별 총 Read 수는 **절대 변경되면 안 됩니다**. 각 샘플 작업은 `.partial` 임시 파일로 작성되고 검증 이후에만 최종 경로에 복사됩니다.


In [ ]:
STATUS=('both_end','single_end','unmatched','ambiguous')
OUTPUT_SUFFIX={
    'both_end':'primer_both_ends_trimmed_unique.csv.gz',
    'single_end':'primer_one_end_trimmed_unique.csv.gz',
    'unmatched':'primer_unmatched_original_unique.csv.gz',
    'ambiguous':'primer_ambiguous_original_unique.csv.gz'
}

def verify_table(path):
    rows=total=0;last=None
    with gzip.open(path,'rt',newline='') as f:
        reader=csv.DictReader(f)
        assert reader.fieldnames==['sequence','read_count','length_bp'],reader.fieldnames
        for row in reader:
            seq=row['sequence'];count=int(row['read_count']);L=int(row['length_bp'])
            assert seq and L==len(seq) and count>0
            if last is not None: assert last<seq,'Unsorted or duplicate sequence: '+seq[:30]
            last=seq;rows+=1;total+=count
    return rows,total

def transform_sample(sample):
    src=BASE/f'{sample}_merged_unique_sequences.csv.gz'
    if not src.exists(): raise FileNotFoundError(src)
    expected={x:OUT/f'{sample}_{OUTPUT_SUFFIX[x]}' for x in STATUS}
    # 모든 기존 결과가 성공적으로 존재하면 재계산 없이 검증해서 재사용
    if not OVERWRITE and all(p.exists() for p in expected.values()):
        validated={x:verify_table(p) for x,p in expected.items()}
        return {'sample':sample,'status':'EXISTS_VERIFIED',
            'input_merged_reads':sum(x[1] for x in validated.values()),
            **{f'{k}_unique':v[0] for k,v in validated.items()},
            **{f'{k}_reads':v[1] for k,v in validated.items()}}
    tmp=WORK/sample
    tmp.mkdir(parents=True,exist_ok=True)
    raw=tmp/'categorized.tsv'; sorted_tsv=tmp/'categorized.sorted.tsv'
    source_rows=source_reads=0
    patterns_unique=collections.Counter();patterns_reads=collections.Counter()
    start=time.monotonic()
    with gzip.open(src,'rt',newline='') as fi,open(raw,'w',buffering=2**20) as fo:
        rd=csv.DictReader(fi)
        assert rd.fieldnames==['sequence','read_count','length_bp'],rd.fieldnames
        for row in rd:
            seq=row['sequence'].upper();count=int(row['read_count']);L=int(row['length_bp'])
            if not seq or len(seq)!=L or count<=0 or any(nt not in 'ACGTN' for nt in seq):
                raise ValueError(f'Invalid input row {source_rows+1}: {sample}')
            category,trimmed,pattern=detect_terminal(seq)
            fo.write(f'{category}\t{trimmed}\t{count}\n')
            patterns_unique[(category,pattern)]+=1
            patterns_reads[(category,pattern)]+=count
            source_rows+=1;source_reads+=count
            if source_rows%500000==0:print(f'[{sample}] {source_rows:,} unique rows scanned',flush=True)
    sort_cmd=['sort','-T',str(tmp),'-S',SORT_RAM,'--parallel=4',
              '-t','\t','-k1,1','-k2,2',str(raw),'-o',str(sorted_tsv)]
    env=os.environ.copy();env['LC_ALL']='C'
    subprocess.run(sort_cmd,check=True,env=env)
    raw.unlink(missing_ok=True)
    out_tmp={key:tmp/(sample+'_'+OUTPUT_SUFFIX[key]) for key in STATUS}
    gz={key:gzip.open(out_tmp[key],'wt',compresslevel=1,newline='') for key in STATUS}
    writers={key:csv.writer(gz[key]) for key in STATUS}
    counts={key:[0,0] for key in STATUS} # [unique, reads]
    try:
        for writer in writers.values():writer.writerow(['sequence','read_count','length_bp'])
        prev=None;sum_count=0
        def emit(key,n):
            kind,seq=key
            writers[kind].writerow((seq,n,len(seq)))
            counts[kind][0]+=1;counts[kind][1]+=n
        with open(sorted_tsv) as f:
            for line in f:
                kind,seq,num=line.rstrip('\n').split('\t')
                key=(kind,seq);num=int(num)
                if prev is not None and key!=prev:
                    emit(prev,sum_count);sum_count=0
                prev=key;sum_count+=num
        if prev is not None:emit(prev,sum_count)
    finally:
        for f in gz.values(): f.close()
    sorted_tsv.unlink(missing_ok=True)
    assert sum(v[1] for v in counts.values())==source_reads, (counts,source_reads)
    assert sum(v[0] for v in counts.values())<=source_rows
    for key,p in out_tmp.items():
        assert tuple(counts[key])==verify_table(p),(key,counts[key])
    # Drive 복사 중단으로 생긴 반쪽 결과가 최종 결과로 보이지 않도록 stage 후 rename
    for key,p in out_tmp.items():
        staged=expected[key].with_name(expected[key].name+'.partial')
        shutil.copyfile(p,staged)
        assert tuple(counts[key])==verify_table(staged)
        staged.replace(expected[key])
    with open(OUT/f'{sample}_primer_patterns.csv','w',newline='') as f:
        w=csv.writer(f);w.writerow(['status','primer_pattern','unique_input_rows','read_count'])
        for (status,pattern),n in sorted(patterns_unique.items()):
            w.writerow([status,pattern,n,patterns_reads[(status,pattern)]])
    summary={'sample':sample,'status':'DONE','input_merged_reads':source_reads,
            'input_unique_sequences':source_rows,'elapsed_sec':round(time.monotonic()-start,1)}
    for key,(u,n) in counts.items():
        summary[f'{key}_unique']=u;summary[f'{key}_reads']=n
    shutil.rmtree(tmp,ignore_errors=True)
    return summary

rows=[]
for sample in SAMPLES:
    print('START',sample,flush=True)
    try:
        r=transform_sample(sample)
        rows.append(r)
        print('DONE',sample,'both/single/unmatched reads =',r.get('both_end_reads'),r.get('single_end_reads'),r.get('unmatched_reads'),flush=True)
    except Exception as ex:
        rows.append({'sample':sample,'status':f'ERROR: {ex}'})
        print('ERROR',sample,repr(ex),flush=True)
    pd.DataFrame(rows).to_csv(OUT/'primer_trimming_11_samples_summary.csv',index=False)
summary=pd.DataFrame(rows)
display(summary)
print('Output folder:',OUT)


## 3. 결과 해석

`both_end`: 양쪽 Primer가 확인된 subset만 strict하게 trimming. `single_end`: 한쪽 Primer만 확실하므로 그쪽만 잘라낸 별도 데이터. `unmatched`: 자르지 않고 원본을 남긴 데이터. `ambiguous`: 잘못 자를 위험이 있어 보존.

⚠️ 따라서 `both_end_unique`만으로 11개 전체의 diversity를 비교하면 selection bias가 생길 수 있습니다. 반드시 `primer_trimming_11_samples_summary.csv`의 **분류별 reads 및 unique 비율**을 확인하세요. 서열이 일부만 포함됐거나 primer에 mismatch가 있으면 이 단계에서 제거되지 않습니다.

P5/P7 어댑터와 6N Index의 정확한 라이브러리 구조를 확보하면 5′ barcode를 안전하게 제거하고 전체 표준화 방향까지 추가할 수 있습니다. **현재 6N을 고정 제거하지 않음.**
